# reduce-op-mean-divide — worked example 3: Weighted mean via two SUM all-reduces

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `reduce-op-mean-divide`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A weighted average across ranks needs two reductions: one over the weighted values `w*x` and one over the weights `w`. The result is `sum(w*x) / sum(w)`. There is no single divide-by-world_size here because the denominator is itself a reduced quantity, not the rank count.

## Worked solution

Each rank contributes a value `x` and a weight `w`. `weighted_mean` packs `[w*x, w]` into one length-2 tensor and `all_reduce(SUM)`s it, so afterward `tensor[0]` is the total weighted value and `tensor[1]` is the total weight. The weighted mean is `tensor[0] / tensor[1]`. Packing both quantities into one tensor means a single collective call instead of two. We print the weighted mean of values [2,8] with weights [3,1]: `(2*3 + 8*1)/(3+1) = 14/4 = 3.5`.

In [ ]:
class FakeDist:
    def __init__(self, per_rank_pairs):
        wx = sum(w * x for x, w in per_rank_pairs)
        wsum = sum(w for _, w in per_rank_pairs)
        self.total = t.tensor([wx, wsum])
    def all_reduce(self, tensor, op='sum'):
        tensor.copy_(self.total)


def weighted_mean(local_x, local_w, dist_module):
    tensor = t.tensor([local_w * local_x, local_w])
    dist_module.all_reduce(tensor, op='sum')
    return (tensor[0] / tensor[1]).item()


pairs = [(2.0, 3.0), (8.0, 1.0)]  # (value, weight) per rank
fd = FakeDist(pairs)
print('weighted mean:', weighted_mean(2.0, 3.0, fd))